# Bias-variance tradeoff via bootstrap (Issue #3)

This notebook studies the bias-variance tradeoff for ordinary least squares
(OLS) regression on noisy samples of the Runge function

$$
f(x) = \frac{1}{1 + 25x^2}, \qquad x \in [-1, 1],
$$

a classic example of a smooth function that is notoriously hard to fit with a
single high-degree polynomial (Runge's phenomenon: oscillations blow up near
the domain edges as the degree grows).

The notebook proceeds in two stages:

1. **Baseline degree sweep** — a single train/test split, sweeping the
   polynomial degree and tracking train/test MSE, reproducing the shape of
   Figure 2.11 in Hastie, Tibshirani & Friedman (*The Elements of Statistical
   Learning*, 2nd ed.).
2. **Bootstrap bias-variance decomposition** — for each degree, bootstrap
   resampling of the training set is used to explicitly estimate the bias$^2$
   and variance terms of the test MSE (ESL eq. 7.9), and how that
   decomposition shifts as the number of data points and the observation
   noise are varied.

All reusable logic (design matrices, OLS, the bootstrap sweep) lives in
`src/fys_stk4155_p1/`; this notebook only orchestrates calls and plots the
results.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from fys_stk4155_p1.data.runge import generate_runge_data, runge_function
from fys_stk4155_p1.regression.degree_sweep import fit_polynomial_degree_sweep
from fys_stk4155_p1.regression.ordinary_least_squares import OLS

In [ ]:
# Generate data
noise_std = 0.1
x, y = generate_runge_data(n=100, noise_std=noise_std, seed=42)


x_plot = np.linspace(-1, 1, 500)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x_plot, runge_function(x_plot), color="black", lw=1.5, label="Runge's function")
ax.scatter(x, y, s=15, alpha=0.6, label=rf"data, $\sigma = {noise_std}$")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Runge function: noisy samples vs. ground truth")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# Baseline experiment: sweep polynomial degree at the original sample size (n=100).
degrees = range(1, 23)
results = fit_polynomial_degree_sweep(x, y, degrees, OLS)

degrees_arr = results["degrees"]
intercept = results["intercept"]
weights = results["weights"]
mse_train, mse_test = results["mse_train"], results["mse_test"]
r2_train, r2_test = results["r2_train"], results["r2_test"]

## 1. Baseline degree sweep

A single 80/20 train/test split, sweeping the polynomial degree from 1 to 22.
Here we reproduce figure 2.11 from Hastie et al.:

In [ ]:
degrees_arr = np.array(list(degrees))

best_idx = np.argmin(mse_test)
best_degree = degrees_arr[best_idx]

fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(
    degrees_arr,
    mse_train,
    marker="o",
    markersize=3,
    label="Train",
)

ax.plot(
    degrees_arr,
    mse_test,
    marker="o",
    markersize=3,
    label="Test",
)

ax.axvline(
    best_degree,
    color="black",
    linestyle="--",
    linewidth=1,
    label=f"Best degree = {best_degree}",
)

ax.set_yscale("log")
ax.set_xlabel("Polynomial degree")
ax.set_ylabel("MSE")
ax.set_title("MSE vs. polynomial degree")
ax.legend()

fig.tight_layout()
plt.show()

We see that the overfitting explodes after degree 20, where the test error increases massively while the test error continues decreasing monotonically. We can also observe that the lowest degrees (0 to around 8) have a higher bias (high loss for both test and train), but a lower variance (both losses are roughly equal). The best polynomial degree appears to be degree 12, where both the train and test error is low.

The cost function of OLS can be written as

$$
C(X, \theta) = \mathbb{E}\left[ (y-\tilde{y})^2 \right] = \text{Bias}^2[\tilde{y}] + \text{Var}[\tilde{y}] + \sigma^2.
$$

## 2. Bootstrap estimate of the bias-variance decomposition

We now estimate bias$^2$ and variance directly via bootstrap resampling of the training set, following Hastie et al. eq. (7.9). For each polynomial degree we hold out a fixed test set, draw `n_bootstraps` bootstrap resamples of the training data, refit OLS on each, and decompose the resulting spread of test predictions into a bias and a variance term. The dotted horizontal line marks $\sigma^2$, the irreducible noise floor that the bias term converges to from above as flexibility increases (since our bias$^2$ estimate is really bias$^2$ + $\sigma^2$, per the cost-function identity above).

In [ ]:
from fys_stk4155_p1.resampling.bootstrap import bootstrap_bias_variance_sweep

degrees_bv = range(0, 14)
bv_results = bootstrap_bias_variance_sweep(x, y, degrees_bv, OLS, n_bootstraps=100, seed=42)
bv_best_degree = bv_results["degrees"][np.argmin(bv_results["mse_test"])]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(bv_results["degrees"], bv_results["mse_test"], "o-", markersize=4, label="test error")
ax.plot(
    bv_results["degrees"],
    bv_results["bias2"],
    "s-",
    markersize=4,
    label=r"bias$^2$ (+ $\sigma^2$)",
)
ax.plot(bv_results["degrees"], bv_results["variance"], "d-", markersize=4, label="variance")
ax.axhline(noise_std**2, color="gray", linestyle=":", linewidth=1.5, label=r"$\sigma^2$")
ax.axvline(
    bv_best_degree,
    color="black",
    linestyle="--",
    linewidth=1,
    label=f"best degree = {bv_best_degree}",
)
ax.set_yscale("log")
ax.set_xlabel("Polynomial degree")
ax.set_ylabel("MSE decomposition")
ax.set_title(rf"Bias-variance tradeoff via bootstrap ($n=100$, $\sigma={noise_std}$)")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

In [ ]:
degrees_list = list(bv_results["degrees"])
for d in (0, 4, 8, degrees_list[-1]):
    idx = degrees_list.index(d)
    print(
        f"degree {d:2d}: error {bv_results['mse_test'][idx]:.4f}  "
        f"bias^2 {bv_results['bias2'][idx]:.4f}  var {bv_results['variance'][idx]:.4f}  "
        f"sum {bv_results['bias2'][idx] + bv_results['variance'][idx]:.4f}"
    )

### Varying the number of data points

More training data should reduce variance at a given polynomial degree (more data pins down the fit), without changing the bias much, since bias is set by the mismatch between the model's expressiveness and the true function.

In [ ]:
ns = [50, 100, 200, 400]

fig, axes = plt.subplots(1, len(ns), figsize=(4 * len(ns), 4), sharey=True)
for ax, n_i in zip(axes, ns, strict=True):
    x_i, y_i = generate_runge_data(n=n_i, noise_std=noise_std, seed=42)
    res_i = bootstrap_bias_variance_sweep(x_i, y_i, degrees_bv, OLS, n_bootstraps=100, seed=42)
    best_degree_i = res_i["degrees"][np.argmin(res_i["mse_test"])]

    ax.plot(res_i["degrees"], res_i["mse_test"], "o-", markersize=3, label="test error")
    ax.plot(res_i["degrees"], res_i["bias2"], "s-", markersize=3, label=r"bias$^2$ (+ $\sigma^2$)")
    ax.plot(res_i["degrees"], res_i["variance"], "d-", markersize=3, label="variance")
    ax.axhline(noise_std**2, color="gray", linestyle=":", linewidth=1.5, label=r"$\sigma^2$")
    ax.axvline(best_degree_i, color="black", linestyle="--", linewidth=1)
    ax.annotate(
        f"best={best_degree_i}",
        xy=(best_degree_i, 1.0),
        xycoords=ax.get_xaxis_transform(),
        xytext=(3, -3),
        textcoords="offset points",
        fontsize=8,
        va="top",
    )

    ax.set_yscale("log")
    ax.set_xlabel("Polynomial degree")
    ax.set_title(f"n = {n_i}")

axes[0].set_ylabel("MSE decomposition")
axes[-1].legend(frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1.0))
fig.suptitle(rf"Effect of sample size on the bias-variance tradeoff ($\sigma={noise_std}$)")
fig.tight_layout()
plt.show()

### Varying the noise level

Higher observation noise ($\sigma$) raises the irreducible error floor uniformly across degrees, and since our bias$^2$ estimate implicitly includes $\sigma^2$ (see the docstring of `bootstrap_bias_variance_sweep`), it should shift the whole curve up without changing where the bias-variance crossover occurs.

In [ ]:
noise_levels = [0.1, 0.2, 0.4, 0.8]

fig, axes = plt.subplots(1, len(noise_levels), figsize=(4 * len(noise_levels), 4), sharey=True)
for ax, noise_i in zip(axes, noise_levels, strict=True):
    x_i, y_i = generate_runge_data(n=100, noise_std=noise_i, seed=42)
    res_i = bootstrap_bias_variance_sweep(x_i, y_i, degrees_bv, OLS, n_bootstraps=100, seed=42)
    best_degree_i = res_i["degrees"][np.argmin(res_i["mse_test"])]

    ax.plot(res_i["degrees"], res_i["mse_test"], "o-", markersize=3, label="test error")
    ax.plot(res_i["degrees"], res_i["bias2"], "s-", markersize=3, label=r"bias$^2$ (+ $\sigma^2$)")
    ax.plot(res_i["degrees"], res_i["variance"], "d-", markersize=3, label="variance")
    ax.axhline(noise_i**2, color="gray", linestyle=":", linewidth=1.5, label=r"$\sigma^2$")
    ax.axvline(best_degree_i, color="black", linestyle="--", linewidth=1)
    ax.annotate(
        f"best={best_degree_i}",
        xy=(best_degree_i, 1.0),
        xycoords=ax.get_xaxis_transform(),
        xytext=(3, -3),
        textcoords="offset points",
        fontsize=8,
        va="top",
    )

    ax.set_yscale("log")
    ax.set_xlabel("Polynomial degree")
    ax.set_title(rf"$\sigma$ = {noise_i}")

axes[0].set_ylabel("MSE decomposition")
axes[-1].legend(frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1.0))
fig.suptitle("Effect of noise level on the bias-variance tradeoff (n=100)")
fig.tight_layout()
plt.show()

TODO: discuss the observed sample-size and noise trends once the cells above have been run.